In [1]:
import numpy as np
import os

import sys
sys.path.append('../')
from Utilities.AncillaryFunctions import FFT_PSD, mu_law_encode

### Functions

In [2]:
def RandSampling(Data, seed=42, frame_length = 10):
    '''
    Randomly samples input data based on local value distributions within frames.

    The input signal is divided into non-overlapping frames of specified length.
    Each frame's value distribution is used to perform random sampling, preserving
    the statistical characteristics of the original data to a controllable degree.

    Args:
        Data (np.ndarray): Input array-like object.
        seed (int, optional): Random seed for reproducibility. Defaults to 42.
        frame_length (int, optional): Length of each frame (segment) for local sampling.
            Shorter frame lengths preserve local features better, while longer frame
            lengths result in smoother, more generalized sampling. Defaults to 20.

    Returns:
        np.ndarray: Sampled data array with the same shape as the input.
    '''
    
    # Set random seed for reproducibility
    np.random.seed(seed)
    
    # Convert all data to numpy array and process
    Data = np.squeeze(Data)
    
    # Create array to store results
    sampled = np.zeros_like(Data)
    total_rows = len(Data)
   
    
    for i, row in enumerate(Data):
        if i % 100 == 0:  # Reduce progress print frequency
            print(f"Progress: {round(i/total_rows*100, 2)}%")
        
        # Create frames using reshape (alternative to tf.signal.frame)
        n_frames = len(row) // frame_length
        frames = row[:n_frames * frame_length].reshape(-1, frame_length)

        
        # Vectorize probability calculation for each frame
        probabilities = np.array([np.bincount(frame, minlength=256) / frame_length for frame in frames])

        
        # Perform sampling for multiple frames at once
        sampled_frames = np.array([np.random.choice(256, size=frame_length, p=prob) for prob in probabilities])

        
        # Store results
        sampled[i, :n_frames * frame_length] = sampled_frames.flatten()
        
        # Process the last frame (if necessary)
        if len(row) % frame_length != 0:
            last_frame = row[n_frames * frame_length:]
            last_prob = np.bincount(last_frame, minlength=256) / len(last_frame)
            sampled[i, n_frames * frame_length:] = np.random.choice(256, size=len(last_frame), p=last_prob)
    
    return sampled[:, None] if len(Data.shape) == 2 else sampled

###  Mu-Law Encoding & Random Sampling 


In [3]:
FileList = [i for i in os.listdir('./ProcessedData/') if 'ART' in i or 'II' in i ]


for sig in FileList[:]:
    sig_name  = sig.split('.')[0]
    print(sig_name)
    print()
    raw_signal = np.load('./ProcessedData/'+ sig)
    Data_mu_law = mu_law_encode(raw_signal[..., None], quantization_channels=256)
    Sampled_Data = np.squeeze(RandSampling(Data_mu_law, seed=42, frame_length=10))[..., None]
    np.save('ProcessedData/Sampled'+sig_name+'.npy', Sampled_Data)
    np.save('ProcessedData/MuLaw'+sig_name+'.npy', Data_mu_law)
    print()


Mimic3TestART

Progress: 0.0%
Progress: 10.0%
Progress: 20.0%
Progress: 30.0%
Progress: 40.0%
Progress: 50.0%
Progress: 60.0%
Progress: 70.0%
Progress: 80.0%
Progress: 90.0%

Mimic3TestII

Progress: 0.0%
Progress: 10.0%
Progress: 20.0%
Progress: 30.0%
Progress: 40.0%
Progress: 50.0%
Progress: 60.0%
Progress: 70.0%
Progress: 80.0%
Progress: 90.0%

Mimic3TrART

Progress: 0.0%
Progress: 1.43%
Progress: 2.86%
Progress: 4.29%
Progress: 5.71%
Progress: 7.14%
Progress: 8.57%
Progress: 10.0%
Progress: 11.43%
Progress: 12.86%
Progress: 14.29%
Progress: 15.71%
Progress: 17.14%
Progress: 18.57%
Progress: 20.0%
Progress: 21.43%
Progress: 22.86%
Progress: 24.29%
Progress: 25.71%
Progress: 27.14%
Progress: 28.57%
Progress: 30.0%
Progress: 31.43%
Progress: 32.86%
Progress: 34.29%
Progress: 35.71%
Progress: 37.14%
Progress: 38.57%
Progress: 40.0%
Progress: 41.43%
Progress: 42.86%
Progress: 44.29%
Progress: 45.71%
Progress: 47.14%
Progress: 48.57%
Progress: 50.0%
Progress: 51.43%
Progress: 52.86%
Progr